# 01 - Explore AEMO prices

First look at Victorian electricity price and demand data for January 2026.

Findings so far:
- 8,928 rows, one per 5-minute interval, no gaps or missing values
- Timestamps are "period ending" (00:05 covers 00:00 to 00:05)
- Converted SETTLEMENTDATE from text to datetime, added hour, day_name and day columns
- Prices are right-skewed: median about $19, mean about $38.55, max over $3,000
- 29% of intervals had negative prices, lowest was -$162.36 at midday on 1 January

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/PRICE_AND_DEMAND_202601_VIC1.csv")

In [3]:
df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
0,VIC1,2026/01/01 00:05:00,4188.23,58.51,TRADE
1,VIC1,2026/01/01 00:10:00,4169.89,25.66,TRADE
2,VIC1,2026/01/01 00:15:00,4143.35,21.31,TRADE
3,VIC1,2026/01/01 00:20:00,4158.40,48.38,TRADE
4,VIC1,2026/01/01 00:25:00,4157.45,47.49,TRADE


In [4]:
df.tail()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
8923,VIC1,2026/01/31 23:40:00,4903.74,19.18,TRADE
8924,VIC1,2026/01/31 23:45:00,4878.79,35.94,TRADE
8925,VIC1,2026/01/31 23:50:00,4856.85,47.89,TRADE
8926,VIC1,2026/01/31 23:55:00,4838.86,49.67,TRADE
8927,VIC1,2026/02/01 00:00:00,4801.49,49.99,TRADE


In [6]:
df.shape

(8928, 5)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8928 entries, 0 to 8927
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   REGION          8928 non-null   str    
 1   SETTLEMENTDATE  8928 non-null   str    
 2   TOTALDEMAND     8928 non-null   float64
 3   RRP             8928 non-null   float64
 4   PERIODTYPE      8928 non-null   str    
dtypes: float64(2), str(3)
memory usage: 593.0 KB


In [8]:
df.tail(3)

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
8925,VIC1,2026/01/31 23:50:00,4856.85,47.89,TRADE
8926,VIC1,2026/01/31 23:55:00,4838.86,49.67,TRADE
8927,VIC1,2026/02/01 00:00:00,4801.49,49.99,TRADE


In [9]:
df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"], format="%Y/%m/%d %H:%M:%S")
df.dtypes

REGION                       str
SETTLEMENTDATE    datetime64[us]
TOTALDEMAND              float64
RRP                      float64
PERIODTYPE                   str
dtype: object

In [10]:
df["hour"] = df["SETTLEMENTDATE"].dt.hour
df["day_name"] = df["SETTLEMENTDATE"].dt.day_name()
df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,hour,day_name
0,VIC1,2026-01-01 00:05:00,4188.23,58.51,TRADE,0,Thursday
1,VIC1,2026-01-01 00:10:00,4169.89,25.66,TRADE,0,Thursday
2,VIC1,2026-01-01 00:15:00,4143.35,21.31,TRADE,0,Thursday
3,VIC1,2026-01-01 00:20:00,4158.40,48.38,TRADE,0,Thursday
4,VIC1,2026-01-01 00:25:00,4157.45,47.49,TRADE,0,Thursday


In [12]:
df["SETTLEMENTDATE"].diff().value_counts()

SETTLEMENTDATE
0 days 00:05:00    8927
Name: count, dtype: int64

In [14]:
df["day"] = df["SETTLEMENTDATE"].dt.day
df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,hour,day_name,day
0,VIC1,2026-01-01 00:05:00,4188.23,58.51,TRADE,0,Thursday,1
1,VIC1,2026-01-01 00:10:00,4169.89,25.66,TRADE,0,Thursday,1
2,VIC1,2026-01-01 00:15:00,4143.35,21.31,TRADE,0,Thursday,1
3,VIC1,2026-01-01 00:20:00,4158.40,48.38,TRADE,0,Thursday,1
4,VIC1,2026-01-01 00:25:00,4157.45,47.49,TRADE,0,Thursday,1


In [15]:
df["RRP"].describe()

count    8928.000000
mean       38.551793
std        89.009769
min      -162.360000
25%        -2.700000
50%        19.180000
75%        63.615000
max      3026.010000
Name: RRP, dtype: float64

In [16]:
df.loc[df["RRP"].idxmax()]

REGION                           VIC1
SETTLEMENTDATE    2026-01-27 17:50:00
TOTALDEMAND                  10658.44
RRP                           3026.01
PERIODTYPE                      TRADE
hour                               17
day_name                      Tuesday
day                                27
Name: 7701, dtype: object

In [17]:
df.sort_values("RRP", ascending=False).head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE,hour,day_name,day
7701,VIC1,2026-01-27 17:50:00,10658.44,3026.01,TRADE,17,Tuesday,27
7700,VIC1,2026-01-27 17:45:00,10600.71,1937.23,TRADE,17,Tuesday,27
7702,VIC1,2026-01-27 17:55:00,10748.28,1703.80,TRADE,17,Tuesday,27
7699,VIC1,2026-01-27 17:40:00,10566.73,1703.42,TRADE,17,Tuesday,27
1970,VIC1,2026-01-07 20:15:00,8011.26,1392.72,TRADE,20,Wednesday,7


In [18]:
negative = df[df["RRP"] < 0]
len(negative)

2555

In [19]:
(df["RRP"] < 0).sum()

np.int64(2555)

In [20]:
df.loc[df["RRP"].idxmin()]

REGION                           VIC1
SETTLEMENTDATE    2026-01-01 12:00:00
TOTALDEMAND                   1458.14
RRP                           -162.36
PERIODTYPE                      TRADE
hour                               12
day_name                     Thursday
day                                 1
Name: 143, dtype: object

In [21]:
(df["RRP"] > 300).sum()

np.int64(83)